In [ ]:
import h5py
import sys

def print_structure(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}, Shape: {obj.shape}, Type: {obj.dtype}")
    else:
        print(f"Group: {name}")

with h5py.File('results/feature_engineering/20250420_153250/selected_features.h5', 'r') as f:
    print("\nH5文件结构:")
    f.visititems(print_structure)

In [ ]:
import h5py
import numpy as np
import pandas as pd

# 替换为您的 H5 文件路径

file_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/MFCAN/src/data/processed/preprocessed_data.h5"
# file_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/MFCAN/src/data/processed/feature_groups.h5"
# 打开 H5 文件并检查结构和标签
with h5py.File(file_path, 'r') as f:
# 打印文件的所有组和数据集
print("H5 文件结构:")
def print_structure(name, obj):
if isinstance(obj, h5py.Dataset):
print(f" 数据集: {name}, 形状: {obj.shape}, 类型: {obj.dtype}")
else:
print(f" 组: {name}")
f.visititems(print_structure)

print("\n检查标签可访问性:")

# 检查各种可能的标签路径
label_paths = [
'labels/train',
'train/labels',
'group_diffusion/train/labels',
'group_qti/train/labels',
'group_cest/train/labels'
]

for path in label_paths:
if path in f:
print(f"✅ 可以访问标签路径: {path}, 形状: {f[path].shape}")
# 显示一些标签值示例
labels = f[path][:]
print(f" 标签值范围: {np.min(labels)} - {np.max(labels)}")
print(f" 唯一标签: {np.unique(labels)[:10]}... (显示前10个)")
else:
print(f"❌ 无法访问标签路径: {path}")

print("\n检查特征组数据:")
modalities = ['diffusion', 'qti', 'cest']
for modality in modalities:
path = f'group_{modality}/train'
if path in f:
print(f"✅ 可以访问 {modality} 特征组: {path}")
# 检查特征数据
if 'features' in f[path]:
print(f" 特征形状: {f[path]['features'].shape}")
else:
print(f" ❌ 特征组中没有 'features' 数据集")
else:
print(f"❌ 无法访问特征组: {path}")

In [ ]:
with h5py.File(file_path, 'r') as f:
    print("特征组训练集大小验证:")
    
    for modality in ['diffusion', 'qti', 'cest']:
        train_path = f'group_{modality}/train/features'
        if train_path in f:
            features = f[train_path][:]
            print(f"{modality} 训练集:")
            print(f"  形状: {features.shape}")
            print(f"  非零值: {np.count_nonzero(features)} / {features.size} ({np.count_nonzero(features) / features.size:.2%})")
            print(f"  平均值: {np.mean(features):.6f}")
            print(f"  标准差: {np.std(features):.6f}")
            print(f"  最小值: {np.min(features):.6f}")
            print(f"  最大值: {np.max(features):.6f}")
        else:
            print(f"❌ 无法访问 {modality} 训练集: {train_path}")

In [ ]:
import h5py
import numpy as np

# 打开原始H5文件
input_file = 'preprocessed_data.h5'
output_file = 'reorganized_encoder_data.h5'

# 索引范围 - 根据您的设计文档
diffusion_indices = slice(0, 15)  # 前15维特征
qti_indices = slice(15, 225)      # 15-224之间的特征
cest_indices = slice(225, 341)    # 225-340之间的特征

with h5py.File(input_file, 'r') as src:
    with h5py.File(output_file, 'w') as dst:
        # 首先复制原始结构
        for split in ['train', 'val', 'test']:
            # 复制特征
            features = src[f'{split}/features'][()]
            dst.create_dataset(f'{split}/features', data=features)
            
            # 复制标签
            labels = src[f'{split}/labels'][()]
            dst.create_dataset(f'{split}/labels', data=labels)
            
            # 在labels组下创建标签副本
            if f'labels/{split}' not in dst:
                dst.create_dataset(f'labels/{split}', data=labels)
        
        # 创建模态特定的特征组
        for modality, indices in [
            ('diffusion', diffusion_indices),
            ('qti', qti_indices), 
            ('cest', cest_indices)
        ]:
            print(f"处理 {modality} 特征组...")
            
            for split in ['train', 'val', 'test']:
                # 提取当前模态的特征
                features = src[f'{split}/features'][:, indices]
                
                # 在group_{modality}/{split}下创建特征数据集
                group_path = f'group_{modality}/{split}'
                dst.create_dataset(f'{group_path}/features', data=features)
                
                # 复制对应的标签
                labels = src[f'{split}/labels'][()]
                dst.create_dataset(f'{group_path}/labels', data=labels)
                
                print(f"  - {split}: 形状 {features.shape}")

print(f"数据已重新组织并保存到 {output_file}")